In [31]:
import urllib.request
import tarfile
import os
import duckdb
import json
from datetime import datetime
import pandas as pd

In [32]:
def formatar_timestamp(unix_timestamp):
    try:
        return datetime.fromtimestamp(int(unix_timestamp)).isoformat() + "Z"
    except:
        return datetime.utcnow().isoformat() + "Z"

def segundos_para_iso8601(segundos):
    m, s = divmod(int(segundos), 60)
    h, m = divmod(m, 60)
    iso = "PT"
    if h > 0: iso += f"{h}H"
    if m > 0: iso += f"{m}M"
    iso += f"{s}S"
    return iso

def obter_verbo_xapi(eventname):
    eventname = str(eventname).lower()
    if "viewed" in eventname: return "http://id.tincanapi.com/verb/viewed"
    if "submitted" in eventname: return "http://activitystrea.ms/schema/1.0/submit"
    if "completed" in eventname: return "http://adlnet.gov/expapi/verbs/completed"
    if "started" in eventname: return "http://adlnet.gov/expapi/verbs/attempted"
    return "http://id.tincanapi.com/verb/interacted"

def construir_arvore_contexto(instanceid, component, sectionid, courseid):
    parents = []
    
    # 1. Atividade (URL no padrão localhost:8000)
    if instanceid not in ['0', 'nan', 'None', '']:
        # Se for nota, forçamos o tipo para 'assessment', pois o adivinhacao.py procura isso!
        tipo = "http://adlnet.gov/expapi/activities/assessment" if component in ['quiz', 'assessment'] else f"http://adlnet.gov/expapi/activities/{component}"
        parents.append({
            "objectType": "Activity", 
            "id": f"http://localhost:8000/mod/{component}/view.php?id={instanceid}",
            "definition": {"type": tipo, "name": {"en": f"{component.capitalize()} (ID: {instanceid})"}}
        })
        
    # 2. Seção (CRÍTICO: URL do section.php e a palavra mágica "Section")
    if sectionid not in ['0', 'nan', 'None', '']:
        parents.append({
            "objectType": "Activity", 
            "id": f"http://localhost:8000/course/section.php?id={sectionid}",
            "definition": {
                "type": "http://id.tincanapi.com/activitytype/section", 
                "name": {"en": f"Course {courseid} Section {sectionid}"} # O adivinhacao.py vai conseguir fazer o rsplit aqui!
            }
        })
        
    # 3. Matéria
    if courseid not in ['0', 'nan', 'None', '']:
        parents.append({
            "objectType": "Activity", 
            "id": f"http://localhost:8000/course/view.php?id={courseid}",
            "definition": {
                "type": "https://w3id.org/xapi/cmi5/activitytype/course", 
                "name": {"en": f"Course {courseid}"}
            }
        })

    contexto = {"category": [{"objectType": "Activity", "id": "http://localhost:8000", "definition": {"type": "http://id.tincanapi.com/activitytype/lms", "name": {"en": "Moodle"}}}]}
    if parents: 
        contexto["parent"] = parents
    return contexto

In [33]:
# 1. URL corrigida com 'id_' no final do timestamp (padrão do Internet Archive para arquivo cru)
url = "https://web.archive.org/web/20210420235203id_/http://research.moodle.org/158/2/export.tar.gz"
arquivo_compactado = "export.tar.gz"
arquivo_verificacao = "export/mdl_logstore_standard_log.csv" 

def baixar_e_extrair_dados():
    if os.path.exists(arquivo_verificacao):
        print("✅ Os dados brutos já estão presentes na pasta. Pulando o download!")
        return

    print(f"📥 Iniciando o download do dataset gigante... (Isso pode demorar dependendo da conexão)")
    try:
        # 2. Criando um disfarce (User-Agent) para não sermos bloqueados
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
        
        # 3. Baixando o arquivo em "pedacinhos" para não travar a memória do computador
        with urllib.request.urlopen(req) as response, open(arquivo_compactado, 'wb') as out_file:
            while chunk := response.read(8192):
                out_file.write(chunk)
                
        print("📦 Download real concluído! Iniciando a extração dos arquivos...")

        # Extrai o .tar.gz
        with tarfile.open(arquivo_compactado, "r:gz") as tar:
            tar.extractall(path=".")
        
        print("🧹 Extração finalizada! Deletando o arquivo compactado para liberar espaço...")
        os.remove(arquivo_compactado)
        
        print("🚀 Tudo pronto! O dataset está descompactado e pronto para o Pandas.")
        
    except Exception as e:
        print(f"❌ Ocorreu um erro durante o processo: {e}")
        # Limpa o arquivo corrompido para não atrapalhar a próxima tentativa
        if os.path.exists(arquivo_compactado):
            os.remove(arquivo_compactado)

# Executa a função
baixar_e_extrair_dados()

✅ Os dados brutos já estão presentes na pasta. Pulando o download!


In [34]:
def extrair_dados_fuzzy():
    print("🦆 1/2 Extraindo Logs de Interação (Com Seções)...")
    pasta_dados = "export/" # Mude para "" se os CSVs estiverem soltos na mesma pasta do notebook
    
    # Logs ordenados por tempo
    query_logs = f"""
        SELECT logs.timecreated, logs.username, logs.eventname, logs.action, 
               logs.component, logs.contextinstanceid, logs.courseid, cm.section AS section
        FROM read_csv_auto('{pasta_dados}mdl_logstore_standard_log.csv', ALL_VARCHAR=TRUE) AS logs
        LEFT JOIN read_csv_auto('{pasta_dados}mdl_course_modules.csv', ALL_VARCHAR=TRUE) AS cm
            ON logs.contextinstanceid = cm.id
        WHERE logs.username NOT IN ('0', '-1', '', 'nan') AND logs.username IS NOT NULL
        ORDER BY CAST(logs.timecreated AS BIGINT) ASC
    """
    df_logs = duckdb.query(query_logs).df()

    print("🦆 2/2 Extraindo Histórico de Notas (Para o Fuzzy Join)...")
    # Notas puras, ordenadas cronologicamente
    query_notas = f"""
        SELECT notas.timemodified AS timecreated, notas.username, 
               notas.rawgrade, notas.rawgrademax
        FROM read_csv_auto('{pasta_dados}mdl_grade_grades_history.csv', ALL_VARCHAR=TRUE) AS notas
        WHERE notas.username NOT IN ('0', '-1', '', 'nan') 
          AND notas.username IS NOT NULL AND notas.rawgrade IS NOT NULL
        ORDER BY CAST(notas.timemodified AS BIGINT) ASC
    """
    df_notas = duckdb.query(query_notas).df()
    
    return df_logs, df_notas

In [35]:
def construir_arvore_contexto(instanceid, component, sectionid, courseid):
    parents = []
    
    # AJUSTE 1: As URLs precisam ser localhost:8000 para imitar o LRS antigo
    if instanceid not in ['0', 'nan', 'None', '']:
        # Se o componente for 'quiz' ou 'grade', forçamos o tipo para 'assessment'
        tipo = "http://adlnet.gov/expapi/activities/assessment" if component in ['quiz', 'grade'] else f"http://adlnet.gov/expapi/activities/{component}"
        parents.append({
            "objectType": "Activity", 
            "id": f"http://localhost:8000/mod/{component}/view.php?id={instanceid}",
            "definition": {"type": tipo, "name": {"en": f"{component.capitalize()} (ID: {instanceid})"}}
        })
        
    if sectionid not in ['0', 'nan', 'None', '']:
        parents.append({
            "objectType": "Activity", 
            "id": f"http://localhost:8000/course/section.php?id={sectionid}",
            # AJUSTE 2: A palavra "Section" exata que o adivinhacao.py procura no rsplit()
            "definition": {"type": "http://id.tincanapi.com/activitytype/section", "name": {"en": f"Course {courseid} Section {sectionid}"}}
        })
        
    if courseid not in ['0', 'nan', 'None', '']:
        parents.append({
            "objectType": "Activity", 
            "id": f"http://localhost:8000/course/view.php?id={courseid}",
            "definition": {"type": "https://w3id.org/xapi/cmi5/activitytype/course", "name": {"en": f"Course {courseid}"}}
        })

    contexto = {"category": [{"objectType": "Activity", "id": "http://localhost:8000", "definition": {"type": "http://id.tincanapi.com/activitytype/lms", "name": {"en": "Moodle"}}}]}
    if parents: contexto["parent"] = parents
    return contexto

def construir_data_lake_fuzzy(df_logs, df_notas):
    print("🧠 Preparando a inteligência temporal (Fuzzy Dictionary)...")
    
    notas_por_aluno = {}
    for _, row in df_notas.iterrows():
        uid = str(row['username'])
        if uid not in notas_por_aluno:
            notas_por_aluno[uid] = []
        notas_por_aluno[uid].append({
            'time': int(row['timecreated']),
            'raw': float(row['rawgrade']),
            'max': float(row['rawgrademax'])
        })

    print("⚙️ Construindo Statements e aplicando o Casamento Temporal...")
    statements = []
    memoria_inicio = {}

    for _, row in df_logs.iterrows():
        userid, eventname = str(row['username']), str(row['eventname'])
        timecreated = int(row['timecreated'])
        courseid, instanceid = str(row['courseid']), str(row['contextinstanceid'])
        component = str(row['component']).replace('mod_', '')
        sectionid = str(row.get('section', 'nan'))
        chave_atividade = (userid, courseid, instanceid)
        acao = str(row['action']).lower()

        if acao == "loggedin":
            acao = "logged in"

        if "started" in eventname or "viewed" in eventname:
            if chave_atividade not in memoria_inicio: 
                memoria_inicio[chave_atividade] = timecreated

        # AJUSTE 3: Usar as URLs base corretas no statement principal
        statement = {
            "actor": {"objectType": "Agent", "account": {"homePage": "http://localhost:8000", "name": userid}},
            "verb": {"id": obter_verbo_xapi(eventname), 
                "display": {
                    "en": acao,
                    "en_US": acao
                    }},
            "object": {
                "objectType": "Activity", 
                "id": f"http://localhost:8000/mod/{component}/view.php?id={instanceid}&course={courseid}",
                "definition": {"name": {"en": f"{component.capitalize()} (ID: {instanceid})"}}
            },
            "timestamp": formatar_timestamp(timecreated),
            "context": {"contextActivities": construir_arvore_contexto(instanceid, component, sectionid, courseid)}
        }

        # LÓGICA DE CONCLUSÃO + FUZZY JOIN
        if "submitted" in eventname or "completed" in eventname:
            statement["verb"] = {"id": "http://adlnet.gov/expapi/verbs/completed", "display": {"en": "completed"}}
            statement["result"] = {}

            if chave_atividade in memoria_inicio:
                tempo = timecreated - memoria_inicio[chave_atividade]
                if tempo >= 0: 
                    statement["result"]["duration"] = segundos_para_iso8601(tempo)
                del memoria_inicio[chave_atividade]

            if userid in notas_por_aluno:
                nota_encontrada = None
                
                for nota in notas_por_aluno[userid]:
                    if nota['time'] >= timecreated:
                        nota_encontrada = nota
                        break

                if nota_encontrada:
                    # Injeta a nota no statement de conclusão
                    statement["result"]["score"] = {
                        "raw": nota_encontrada["raw"],
                        "max": nota_encontrada["max"]
                    }
                    
                    # AJUSTE 4: Se nós colamos uma nota nessa atividade, ela TEM que ser um assessment!
                    # Forçamos a reescrita do contexto para garantir que o adivinhacao.py leia essa nota.
                    statement["context"]["contextActivities"] = construir_arvore_contexto(instanceid, 'quiz', sectionid, courseid)
                    
                    notas_por_aluno[userid].remove(nota_encontrada)

            if not statement["result"]:
                del statement["result"]

        statements.append(statement)

    caminho_output = "xapi_statements_completos.json"
    with open(caminho_output, "w", encoding="utf-8") as f:
        json.dump(statements, f, indent=4, ensure_ascii=False)
    print(f"🚀 Sucesso Absoluto! {len(statements):,} statements gerados unindo nota e tempo no padrão xAPI.")

df_logs_fuzzy, df_notas_fuzzy = extrair_dados_fuzzy()
construir_data_lake_fuzzy(df_logs_fuzzy, df_notas_fuzzy)

🦆 1/2 Extraindo Logs de Interação (Com Seções)...
🦆 2/2 Extraindo Histórico de Notas (Para o Fuzzy Join)...
🧠 Preparando a inteligência temporal (Fuzzy Dictionary)...
⚙️ Construindo Statements e aplicando o Casamento Temporal...
🚀 Sucesso Absoluto! 2,391,762 statements gerados unindo nota e tempo no padrão xAPI.


In [36]:
import json

def inspecionar_data_lake():
    caminho_arquivo = "xapi_statements_completos.json"
    print(f"🔍 Abrindo o Data Lake '{caminho_arquivo}' para inspeção...\n")

    try:
        with open(caminho_arquivo, "r", encoding="utf-8") as f:
            statements = json.load(f)
            
        print(f"📊 Total de statements no arquivo: {len(statements):,}".replace(",", "."))
        
        print("\n" + "="*60)
        print("👀 EXEMPLOS DE INTERAÇÃO COMUM (Primeiros registros)")
        print("="*60)
        
        # Exibe os 2 primeiros registros
        for i in range(min(2, len(statements))):
            print(json.dumps(statements[i], indent=4, ensure_ascii=False))
            print("-" * 60)

        print("\n" + "="*60)
        print("🎯 EXEMPLO DE CONCLUSÃO (Com Nota e Tempo de Resposta)")
        print("="*60)
        
        # Busca o primeiro statement que contenha o bloco "result" (nota/tempo)
        statement_com_resultado = next((s for s in statements if "result" in s), None)
        
        if statement_com_resultado:
            print(json.dumps(statement_com_resultado, indent=4, ensure_ascii=False))
        else:
            print("Nenhum statement com 'result' encontrado. Verifique se as notas foram integradas.")
            
    except FileNotFoundError:
        print(f"❌ Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")
    except Exception as e:
        print(f"❌ Erro ao ler o JSON: {e}")

# Executa a inspeção
inspecionar_data_lake()

🔍 Abrindo o Data Lake 'xapi_statements_completos.json' para inspeção...

📊 Total de statements no arquivo: 2.391.762

👀 EXEMPLOS DE INTERAÇÃO COMUM (Primeiros registros)
{
    "actor": {
        "objectType": "Agent",
        "account": {
            "homePage": "http://localhost:8000",
            "name": "user6442803380426375169"
        }
    },
    "verb": {
        "id": "http://id.tincanapi.com/verb/interacted",
        "display": {
            "en": "logged in",
            "en_US": "logged in"
        }
    },
    "object": {
        "objectType": "Activity",
        "id": "http://localhost:8000/mod/core/view.php?id=0&course=0",
        "definition": {
            "name": {
                "en": "Core (ID: 0)"
            }
        }
    },
    "timestamp": "2014-12-04T12:32:49Z",
    "context": {
        "contextActivities": {
            "category": [
                {
                    "objectType": "Activity",
                    "id": "http://localhost:8000",
           

In [37]:
import shutil
import os

# Caminhos absolutos exatos (o 'r' na frente evita problemas com as barras do Windows)
origem = r'C:\Users\Danilus04\Documents\Faculdade\TCC\learning-analytics\transformador\xapi_statements_completos.json'
destino_pasta = r'C:\Users\Danilus04\Documents\Faculdade\TCC\learning-analytics\gerador-de-medida\data'
destino_arquivo = os.path.join(destino_pasta, 'xapi_statements_completos.json')

def mover_resultado():
    # Verificar se o arquivo de origem existe
    if not os.path.exists(origem):
        print(f"❌ Erro: O arquivo de origem não foi encontrado em:\n{origem}")
        return

    # Criar a pasta de destino se ela não existir
    if not os.path.exists(destino_pasta):
        os.makedirs(destino_pasta)
        print(f"📁 Pasta de destino criada: {destino_pasta}")

    try:
        # Mover o arquivo
        shutil.move(origem, destino_arquivo)
        print(f"✅ Sucesso! Arquivo movido para:\n{destino_arquivo}")
    except Exception as e:
        print(f"❌ Erro ao mover o arquivo: {e}")

mover_resultado()

✅ Sucesso! Arquivo movido para:
C:\Users\Danilus04\Documents\Faculdade\TCC\learning-analytics\gerador-de-medida\data\xapi_statements_completos.json
